In [136]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

import seaborn as sns

In [137]:
df = pd.read_csv('tested_fare_imputed.csv')[['Age','Pclass','SibSp','Parch','Survived']]

In [138]:
df.head()

,Age,Pclass,SibSp,Parch,Survived
0,34.5,3,0,0,0
1,47.0,3,1,0,1
2,62.0,2,0,0,0
3,27.0,3,0,0,0
4,22.0,3,1,1,1


In [139]:
df.dropna(inplace=True)

In [140]:
df.head()

,Age,Pclass,SibSp,Parch,Survived
0,34.5,3,0,0,0
1,47.0,3,1,0,1
2,62.0,2,0,0,0
3,27.0,3,0,0,0
4,22.0,3,1,1,1


In [141]:
X = df.iloc[:,0:4]
y = df.iloc[:,-1]

In [142]:
X.head()

,Age,Pclass,SibSp,Parch
0,34.5,3,0,0
1,47.0,3,1,0
2,62.0,2,0,0
3,27.0,3,0,0
4,22.0,3,1,1


In [143]:
np.mean(cross_val_score(LogisticRegression(),X,y,scoring='accuracy',cv=20))

np.float64(0.6238970588235293)

## Applying Feature Construction

In [144]:
X['Family_size'] = X['SibSp'] + X['Parch'] + 1

In [145]:
X.head()

,Age,Pclass,SibSp,Parch,Family_size
0,34.5,3,0,0,1
1,47.0,3,1,0,2
2,62.0,2,0,0,1
3,27.0,3,0,0,1
4,22.0,3,1,1,3


In [146]:
def myfunc(num):
    if num == 1:
        #alone
        return 0
    elif num >1 and num <=4:
        # small family
        return 1
    else:
        # large family
        return 2

In [147]:
myfunc(5)

2

In [148]:
X['Family_type'] = X['Family_size'].apply(myfunc)

In [149]:
X.head()

,Age,Pclass,SibSp,Parch,Family_size,Family_type
0,34.5,3,0,0,1,0
1,47.0,3,1,0,2,1
2,62.0,2,0,0,1,0
3,27.0,3,0,0,1,0
4,22.0,3,1,1,3,1


In [150]:
X.drop(columns=['SibSp','Parch','Family_size'],inplace=True)

In [151]:
X.head()

,Age,Pclass,Family_type
0,34.5,3,0
1,47.0,3,1
2,62.0,2,0
3,27.0,3,0
4,22.0,3,1


In [152]:
np.mean(cross_val_score(LogisticRegression(),X,y,scoring='accuracy',cv=20))

np.float64(0.6056985294117647)

## Feature Splitting

In [153]:
df = pd.read_csv('tested_fare_imputed.csv')

In [154]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [155]:
df['Name']

0                                  Kelly, Mr. James
1                  Wilkes, Mrs. James (Ellen Needs)
2                         Myles, Mr. Thomas Francis
3                                  Wirz, Mr. Albert
4      Hirvonen, Mrs. Alexander (Helga E Lindqvist)
                           ...                     
413                              Spector, Mr. Woolf
414                    Oliva y Ocana, Dona. Fermina
415                    Saether, Mr. Simon Sivertsen
416                             Ware, Mr. Frederick
417                        Peter, Master. Michael J
Name: Name, Length: 418, dtype: str

In [156]:
# Extract the passenger's title (e.g., Mr, Mrs, Miss, Dr) from the Name column
# Step 1: Split the name at ', ' and take the second part
#         Example: "Braund, Mr. Owen Harris" -> "Mr. Owen Harris"
# Step 2: Split that part at '.' and take the first part
#         Example: "Mr. Owen Harris" -> "Mr"
df['Title'] = df['Name'].str.split(', ', expand=True)[1].str.split('.', expand=True)[0]
df['Title']

0          Mr
1         Mrs
2          Mr
3          Mr
4         Mrs
        ...  
413        Mr
414      Dona
415        Mr
416        Mr
417    Master
Name: Title, Length: 418, dtype: str

In [157]:
# Extract only the passenger's actual name (remove the surname and title)
# Step 1: Split the name at ', ' and take the second part
#         Example: "Braund, Mr. Owen Harris" -> "Mr. Owen Harris"
# Step 2: Split that part at '.' and take the second part
#         Example: "Mr. Owen Harris" -> " Owen Harris"
df['Name'] = df['Name'].str.split(', ', expand=True)[1].str.split('.', expand=True)[1]
df['Name']

0                               James
1                 James (Ellen Needs)
2                      Thomas Francis
3                              Albert
4       Alexander (Helga E Lindqvist)
                    ...              
413                             Woolf
414                           Fermina
415                   Simon Sivertsen
416                         Frederick
417                         Michael J
Name: Name, Length: 418, dtype: str

In [158]:
df[['Title','Name']]

,Title,Name
0,Mr,James
1,Mrs,James (Ellen Needs)
2,Mr,Thomas Francis
3,Mr,Albert
4,Mrs,Alexander (Helga E Lindqvist)
...,...,...
413,Mr,Woolf
414,Dona,Fermina
415,Mr,Simon Sivertsen
416,Mr,Frederick


In [159]:
# Group the dataset by passenger Title (Mr, Mrs, Miss, etc.)
# Calculate the average survival rate for each title
# Sort the titles from highest survival rate to lowest
(df.groupby('Title')['Survived']
   .mean()
   .sort_values(ascending=False))

Title
Dona      1.0
Miss      1.0
Mrs       1.0
Ms        1.0
Col       0.0
Dr        0.0
Master    0.0
Mr        0.0
Rev       0.0
Name: Survived, dtype: float64

In [160]:
# Create a new column and initialize all values to 0 (Not Married)
df['Is_Married'] = 0

# Set Is_Married to 1 for passengers whose title is 'Mrs'
df.loc[df['Title'] == 'Mrs', 'Is_Married'] = 1

In [161]:
df['Is_Married']

0      0
1      1
2      0
3      0
4      1
      ..
413    0
414    0
415    0
416    0
417    0
Name: Is_Married, Length: 418, dtype: int64